# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: K-means** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
from pcamarillor.spark_utils import SparkUtils
su = SparkUtils("ML: K-means", 
                "spark://spark-master:7077")
su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 00:59:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Example 1: Clustering with 2D points

In [2]:
# Sample data in Python (e.g., 2D points)
data = [
    (0, 1.0, 1.0),
    (1, 2.0, 1.0),
    (2, 4.0, 5.0),
    (3, 5.0, 5.0),
    (4, 10.0, 10.0),
    (5, 12.0, 11.0)
]

# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("id", "int"), ("x", "float"), ("y", "float")])

# Create DataFrame for k means
random_points_df = su.spark.createDataFrame(data, schema)

## Assemble the features into a single vector column

In [3]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["x", "y"], outputCol="features")
assembled_df = assembler.transform(random_points_df)

## Configure K-means

In [4]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans().setK(3).setSeed(73)

## Train model

In [5]:
model = kmeans.fit(assembled_df)
print("K-means model trained successfully")
kmeans_model_path = "/opt/spark/work-dir/data/mlmodels/kmeans/2D"
model.write().overwrite().save(kmeans_model_path)
model.__class__

26/04/28 01:00:16 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


K-means model trained successfully


pyspark.ml.clustering.KMeansModel

## Get Predictions

In [6]:
from pyspark.ml.clustering import KMeansModel
k_model = KMeansModel.load(kmeans_model_path)
predictions = k_model.transform(assembled_df)

## Evaluate model

In [7]:
from pyspark.ml.evaluation import ClusteringEvaluator

# Evaluate clustering by computing Silhouette score
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette score: {silhouette}")

# Show the result
print("Cluster Centers: ")
for center in model.clusterCenters():
    print(center)

Silhouette score: 0.9494652547284126
Cluster Centers: 
[4.5 5. ]
[11.  10.5]
[1.5 1. ]


# Lab 13: Clustering Wine dataset with K-means

In [8]:
# Downlod dataset from https://www.kaggle.com/datasets/harrywang/wine-dataset-for-clustering

columns_types = [("Alcohol", "float"),
                                     ("Malic_Acid", "float"),
                                     ("Ash", "float"),
                                     ("Ash_Alcanity", "float"),
                                     ("Magnesium", "float"),
                                     ("Total_Phenols", "float"),
                                     ("Flavanoids", "float"),
                                     ("Nonflavanoid_Phenols", "float"),
                                     ("Proanthocyanins", "float"),
                                     ("Color_Intensity", "float"),
                                     ("Hue", "float"),
                                     ("OD280", "float"),
                                     ("Proline", "float")]

# Define schema for the DataFrame
wines_schema = SparkUtils.generate_schema(columns_types)

# Create DataFrame from wines csv
wines_df = su.spark \
                    .read \
                    .option("header", "true") \
                    .schema(wines_schema) \
                    .csv("/opt/spark/work-dir/data/ml/kmeans")


assembler = VectorAssembler(inputCols=[x for x,_ in columns_types], outputCol="features")
assembled_df = assembler.transform(wines_df)

# TODO: Find the optimial K
# TODO: Add the code here to iterate from k = 2, 4, .., 10 and get the silhouette score for each k
max_silhouette = float("-inf")

for k in range(2, 11, 2):
    print("------------------------------------")
    print(f"Training K-means model for k={k} clusters")
    kmeans = KMeans().setK(k).setSeed(13)

    model = kmeans.fit(assembled_df)
    print(f"K-means model trained successfully for {k} clusters")

    # Evaluate clustering by computing Silhouette score
    evaluator = ClusteringEvaluator()
    predictions = model.transform(assembled_df)
    silhouette = evaluator.evaluate(predictions)
    print(f"Silhouette score: {silhouette}")
    
    if silhouette > max_silhouette:
        max_silhouette = silhouette
        best_k = k
print("------------------------------------")
print(f"Best k: {best_k}, Max Silhouette score: {max_silhouette}")


------------------------------------
Training K-means model for k=2 clusters


26/04/28 01:00:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


K-means model trained successfully for 2 clusters
Silhouette score: 0.8193526758797327
------------------------------------
Training K-means model for k=4 clusters
K-means model trained successfully for 4 clusters
Silhouette score: 0.7303066563621069
------------------------------------
Training K-means model for k=6 clusters
K-means model trained successfully for 6 clusters
Silhouette score: 0.7358902017475427
------------------------------------
Training K-means model for k=8 clusters
K-means model trained successfully for 8 clusters
Silhouette score: 0.6986940310339239
------------------------------------
Training K-means model for k=10 clusters
K-means model trained successfully for 10 clusters
Silhouette score: 0.6914941969954019
------------------------------------
Best k: 2, Max Silhouette score: 0.8193526758797327


In [10]:
su.spark.stop()